# Lesson 2: Tool Calling

Use Gemini function calling (via LlamaIndex) to choose tools **and** fill in tool arguments.

## Setup

In [1]:
from helper import get_google_api_key, configure_settings

GOOGLE_API_KEY = get_google_api_key()
llm, embed_model = configure_settings()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [2]:
import nest_asyncio

nest_asyncio.apply()

## 1. Define a Simple Tool

In [3]:
from llama_index.core.tools import FunctionTool

def add(x: int, y: int) -> int:
    """Adds two integers together."""
    return x + y


def mystery(x: int, y: int) -> int:
    """Mystery function that operates on top of two numbers."""
    return (x + y) * (x + y)


add_tool = FunctionTool.from_defaults(fn=add)
mystery_tool = FunctionTool.from_defaults(fn=mystery)

In [4]:
from llama_index.llms.google_genai import GoogleGenAI
from helper import DEFAULT_LLM_MODEL

llm = GoogleGenAI(model=DEFAULT_LLM_MODEL, api_key=GOOGLE_API_KEY, temperature=0)
response = llm.predict_and_call(
    [add_tool, mystery_tool],
    "Tell me the output of the mystery function on 2 and 9",
    verbose=True,
)
print(str(response))

Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/asyncio/events.py", line 88, in _run
    self._context.run(self._callback, *self._args)
RuntimeError: cannot enter context: <_contextvars.Context object at 0x107904400> is already entered


=== Calling Function ===
Calling function: mystery with args: {"x": 2, "y": 9}
=== Function Output ===
121
121


## 2. Define an Auto-Retrieval Tool

### Load Data

```bash
curl -L "https://openreview.net/pdf?id=VtmBAGCN7o" -o metagpt.pdf
```

In [6]:
# !curl -L "https://openreview.net/pdf?id=VtmBAGCN7o" -o metagpt.pdf

from helper import load_pdf

documents = load_pdf("metagpt.pdf")


Task was destroyed but it is pending!
task: <Task pending name='Task-4' coro=<_async_in_context.<locals>.run_in_context() running at /Users/worawatlawanont/Projects/sut-ai-sw-dev/.venv/lib/python3.12/site-packages/ipykernel/utils.py:60> wait_for=<Task pending name='Task-7' coro=<Kernel.shell_main() running at /Users/worawatlawanont/Projects/sut-ai-sw-dev/.venv/lib/python3.12/site-packages/ipykernel/kernelbase.py:597> cb=[Task.__wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at /Users/worawatlawanont/Projects/sut-ai-sw-dev/.venv/lib/python3.12/site-packages/zmq/eventloop/zmqstream.py:563]>
<frozen abc>:105: RuntimeWarning: coroutine 'Kernel.shell_main' was never awaited
Task was destroyed but it is pending!
task: <Task pending name='Task-7' coro=<Kernel.shell_main() running at /Users/worawatlawanont/Projects/sut-ai-sw-dev/.venv/lib/python3.12/site-packages/ipykernel/kernelbase.py:597> cb=[Task.__wakeup()]>


In [7]:
from llama_index.core.node_parser import SentenceSplitter

splitter = SentenceSplitter(chunk_size=1024)
nodes = splitter.get_nodes_from_documents(documents)

In [8]:
print(nodes[0].get_content(metadata_mode="all"))

page_label: 1
file_name: metagpt.pdf
file_path: metagpt.pdf
file_type: application/pdf
file_size: 16911937
creation_date: 2026-07-13
last_modified_date: 2026-07-13

Preprint
METAGPT: M ETA PROGRAMMING FOR A
MULTI-AGENT COLLABORATIVE FRAMEWORK
Sirui Hong1∗, Mingchen Zhuge2∗, Jonathan Chen1, Xiawu Zheng3, Yuheng Cheng4,
Ceyao Zhang4, Jinlin Wang1, Zili Wang, Steven Ka Shing Yau5, Zijuan Lin4,
Liyang Zhou6, Chenyu Ran1, Lingfeng Xiao1,7, Chenglin Wu1†, J¨urgen Schmidhuber2,8
1DeepWisdom, 2AI Initiative, King Abdullah University of Science and Technology,
3Xiamen University, 4The Chinese University of Hong Kong, Shenzhen,
5Nanjing University, 6University of Pennsylvania,
7University of California, Berkeley, 8The Swiss AI Lab IDSIA/USI/SUPSI
ABSTRACT
Remarkable progress has been made on automated problem solving through so-
cieties of agents based on large language models (LLMs). Existing LLM-based
multi-agent systems can already solve simple dialogue tasks. Solutions to more
complex tasks,

In [9]:
from llama_index.core import VectorStoreIndex

vector_index = VectorStoreIndex(nodes)
query_engine = vector_index.as_query_engine(similarity_top_k=2)

In [10]:
from llama_index.core.vector_stores import MetadataFilters

query_engine = vector_index.as_query_engine(
    similarity_top_k=2,
    filters=MetadataFilters.from_dicts(
        [
            {"key": "page_label", "value": "2"},
        ]
    ),
)

response = query_engine.query(
    "What are some high-level results of MetaGPT?",
)

2026-07-13 22:03:20,298 - INFO - AFC is enabled with max remote calls: 10.
2026-07-13 22:03:33,957 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemma-4-31b-it:generateContent "HTTP/1.1 200 OK"


In [11]:
print(str(response))

MetaGPT achieved a new state-of-the-art (SoTA) in code generation benchmarks with Pass@1 scores of 85.9% and 87.7%. In experimental evaluations, it reached a 100% task completion rate, demonstrating robustness and efficiency regarding token and time costs. Additionally, compared to other popular frameworks such as AgentVerse, ChatDev, LangChain, and AutoGPT, MetaGPT stands out in its ability to offer extensive functionality and handle higher levels of software complexity.


In [12]:
for n in response.source_nodes:
    print(n.metadata)

{'page_label': '2', 'file_name': 'metagpt.pdf', 'file_path': 'metagpt.pdf', 'file_type': 'application/pdf', 'file_size': 16911937, 'creation_date': '2026-07-13', 'last_modified_date': '2026-07-13'}


### Define the Auto-Retrieval Tool

In [13]:
from typing import List

from llama_index.core.vector_stores import FilterCondition


def vector_query(
    query: str,
    page_numbers: List[str],
) -> str:
    """Perform a vector search over an index.

    query (str): the string query to be embedded.
    page_numbers (List[str]): Filter by set of pages. Leave BLANK if we want
        to perform a vector search over all pages. Otherwise, filter by the
        set of specified pages.
    """
    metadata_dicts = [
        {"key": "page_label", "value": p} for p in page_numbers
    ]

    query_engine = vector_index.as_query_engine(
        similarity_top_k=2,
        filters=MetadataFilters.from_dicts(
            metadata_dicts,
            condition=FilterCondition.OR,
        ),
    )
    response = query_engine.query(query)
    return response


vector_query_tool = FunctionTool.from_defaults(
    name="vector_tool",
    fn=vector_query,
)

In [14]:
llm = GoogleGenAI(model=DEFAULT_LLM_MODEL, api_key=GOOGLE_API_KEY, temperature=0)
response = llm.predict_and_call(
    [vector_query_tool],
    "What are the high-level results of MetaGPT as described on page 2?",
    verbose=True,
)
print(str(response))

2026-07-13 22:04:13,680 - INFO - HTTP Request: GET https://generativelanguage.googleapis.com/v1beta/models/gemma-4-31b-it "HTTP/1.1 200 OK"
2026-07-13 22:04:18,591 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemma-4-31b-it:generateContent "HTTP/1.1 200 OK"


=== Calling Function ===
Calling function: vector_tool with args: {"query": "high-level results of MetaGPT", "page_numbers": ["2"]}


2026-07-13 22:04:18,915 - INFO - AFC is enabled with max remote calls: 10.
2026-07-13 22:04:40,338 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemma-4-31b-it:generateContent "HTTP/1.1 200 OK"


=== Function Output ===
MetaGPT achieved a new state-of-the-art (SoTA) in code generation benchmarks with Pass@1 scores of 85.9% and 87.7%. In experimental evaluations, it reached a 100% task completion rate, demonstrating efficiency in token and time costs as well as robustness. Additionally, MetaGPT stands out from other frameworks—such as LangChain, AutoGPT, ChatDev, and AgentVerse—by offering extensive functionality and the ability to handle higher levels of software complexity.
MetaGPT achieved a new state-of-the-art (SoTA) in code generation benchmarks with Pass@1 scores of 85.9% and 87.7%. In experimental evaluations, it reached a 100% task completion rate, demonstrating efficiency in token and time costs as well as robustness. Additionally, MetaGPT stands out from other frameworks—such as LangChain, AutoGPT, ChatDev, and AgentVerse—by offering extensive functionality and the ability to handle higher levels of software complexity.


In [15]:
for n in response.source_nodes:
    print(n.metadata)

{'page_label': '2', 'file_name': 'metagpt.pdf', 'file_path': 'metagpt.pdf', 'file_type': 'application/pdf', 'file_size': 16911937, 'creation_date': '2026-07-13', 'last_modified_date': '2026-07-13'}


## Let's add some other tools!

In [20]:
from llama_index.core import SummaryIndex
from llama_index.core.tools import QueryEngineTool

summary_index = SummaryIndex(nodes)
summary_query_engine = summary_index.as_query_engine(
    response_mode="tree_summarize",
    use_async=False,
)
summary_tool = QueryEngineTool.from_defaults(
    name="summary_tool",
    query_engine=summary_query_engine,
    description="Useful if you want to get a summary of MetaGPT",
)


2026-07-13 22:06:55,082 - ERROR - Task was destroyed but it is pending!
task: <Task pending name='Task-107' coro=<_async_in_context.<locals>.run_in_context() done, defined at /Users/worawatlawanont/Projects/sut-ai-sw-dev/.venv/lib/python3.12/site-packages/ipykernel/utils.py:57> wait_for=<Task pending name='Task-108' coro=<Kernel.shell_main() running at /Users/worawatlawanont/Projects/sut-ai-sw-dev/.venv/lib/python3.12/site-packages/ipykernel/kernelbase.py:597> cb=[Task.__wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at /Users/worawatlawanont/Projects/sut-ai-sw-dev/.venv/lib/python3.12/site-packages/zmq/eventloop/zmqstream.py:563]>
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/collections/__init__.py:449: RuntimeWarning: coroutine 'Kernel.shell_main' was never awaited
  result = tuple_new(cls, iterable)
2026-07-13 22:06:55,085 - ERROR - Task was destroyed but it is pending!
task: <Task pending name='Task-108' coro=<Kernel.shell_main() running at /Users

In [21]:
response = llm.predict_and_call(
    [vector_query_tool, summary_tool],
    "What are the MetaGPT comparisons with ChatDev described on page 8?",
    verbose=True,
)
print(str(response))

2026-07-13 22:06:55,096 - ERROR - Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/asyncio/events.py", line 88, in _run
    self._context.run(self._callback, *self._args)
RuntimeError: cannot enter context: <_contextvars.Context object at 0x107904400> is already entered
2026-07-13 22:06:55,096 - ERROR - Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/asyncio/events.py", line 88, in _run
    self._context.run(self._callback, *self._args)
RuntimeError: cannot enter context: <_contextvars.Context object at 0x107904400> is already entered
2026-07-13 22:07:05,152 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemma-4-31b-it:generateContent "HTTP/1.1 200 OK"
2026-07-13 22:07:05,158 - ERROR - Task was dest

=== Calling Function ===
Calling function: vector_tool with args: {"page_numbers": ["8"], "query": "MetaGPT comparisons with ChatDev"}


2026-07-13 22:07:05,392 - INFO - AFC is enabled with max remote calls: 10.
2026-07-13 22:07:33,368 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemma-4-31b-it:generateContent "HTTP/1.1 200 OK"


=== Function Output ===
MetaGPT outperforms ChatDev on the SoftwareDev dataset across nearly all metrics. Key comparisons include:

*   **Executability:** MetaGPT achieves a score of 3.75, while ChatDev scores 2.25.
*   **Running Time:** MetaGPT is faster, taking 503 seconds compared to ChatDev's 762 seconds.
*   **Token Usage and Productivity:** Although MetaGPT uses more total tokens (24,613 or 31,255) than ChatDev (19,292), it is more efficient in code generation, requiring only 124.3 to 126.5 tokens per line of code, whereas ChatDev uses 248.9 tokens.
*   **Code Statistics:** MetaGPT produces more code files (4.6 to 5.1 versus 1.9) and a higher number of total code lines (194.6 to 251.4 versus 77.5).
*   **Human Revision Cost:** MetaGPT has a lower human revision cost (0.83 to 2.25) compared to ChatDev's 2.5.
MetaGPT outperforms ChatDev on the SoftwareDev dataset across nearly all metrics. Key comparisons include:

*   **Executability:** MetaGPT achieves a score of 3.75, while Chat

In [22]:
for n in response.source_nodes:
    print(n.metadata)

{'page_label': '8', 'file_name': 'metagpt.pdf', 'file_path': 'metagpt.pdf', 'file_type': 'application/pdf', 'file_size': 16911937, 'creation_date': '2026-07-13', 'last_modified_date': '2026-07-13'}


In [23]:
response = llm.predict_and_call(
    [vector_query_tool, summary_tool],
    "What is a summary of the paper?",
    verbose=True,
)
print(str(response))

2026-07-13 22:10:05,642 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemma-4-31b-it:generateContent "HTTP/1.1 200 OK"
2026-07-13 22:10:05,679 - INFO - AFC is enabled with max remote calls: 10.


=== Calling Function ===
Calling function: summary_tool with args: {"input": "MetaGPT"}


2026-07-13 22:10:36,683 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemma-4-31b-it:generateContent "HTTP/1.1 200 OK"


=== Function Output ===
MetaGPT is a meta-programming framework designed for multi-agent collaboration based on Large Language Models (LLMs). It incorporates efficient human workflows by encoding Standardized Operating Procedures (SOPs) into prompt sequences, which streamlines workflows and allows agents with domain expertise to verify intermediate results and reduce errors.

The framework utilizes an assembly line paradigm to assign diverse roles to various agents, such as Product Manager, Architect, Project Manager, Engineer, and QA Engineer. Unlike systems that rely on unconstrained natural language dialogue, MetaGPT requires agents to generate structured outputs, including Product Requirements Documents (PRDs), design artifacts, flowcharts, and interface specifications.

Key technical features of MetaGPT include:
*   **Communication Protocol:** To enhance efficiency and avoid information overload, MetaGPT uses a shared global message pool and a publish-subscribe mechanism. This all